# Task 8: RAG with Llama 3.2

A Retrieval-Augmented Generation system for an academic document using Sentence Transformers and Ollama.

In [ ]:
%pip install -q pypdf sentence-transformers ollama scikit-learn

In [ ]:
from pathlib import Path
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import ollama

MODEL = 'llama3.2'
DOCUMENT_PATH = Path('academic_document.pdf')

def load_document(path):
    if path.exists():
        pages = []
        for page in PdfReader(str(path)).pages:
            pages.append(page.extract_text() or '')
        document = '\n'.join(pages).strip()
        if document:
            return document
        print('The PDF contains no extractable text. Using sample text.')
    else:
        print('PDF not found. Using sample academic text.')
    return ('Academic sample: students need at least 75 percent attendance for semester examinations. '
            'The curriculum includes Python, data structures, databases, operating systems, mathematics, '
            'machine learning, and artificial intelligence.')

text = load_document(DOCUMENT_PATH)
print('Characters:', len(text))

In [ ]:
def chunk_text(text, size=800, overlap=150):
    cleaned = ' '.join(text.split())
    if not cleaned:
        return ['No document text was available.']
    step = size - overlap
    return [cleaned[i:i + size] for i in range(0, len(cleaned), step)]

chunks = chunk_text(text)
embedder = SentenceTransformer('all-MiniLM-L6-v2')
vectors = embedder.encode(chunks, normalize_embeddings=True)
print('Chunks:', len(chunks))

In [ ]:
question = input('Enter your academic question: ').strip()
if not question:
    question = 'What is the minimum attendance required?'

q_vector = embedder.encode([question], normalize_embeddings=True)
scores = cosine_similarity(q_vector, vectors)[0]
top_k = min(3, len(chunks))
indices = np.argsort(scores)[::-1][:top_k]
retrieved = [(int(i), float(scores[i]), chunks[i]) for i in indices]
context = '\n\n'.join(f'[Score {score:.3f}] {chunk}' for _, score, chunk in retrieved)

print('\nUSER QUESTION\n', question)
print('\nRETRIEVED CONTEXT\n', context)

In [ ]:
prompt = ('Answer the question using only the context below. If the answer is not present, say that it is not available in the context.\n\n'
          f'Context:\n{context}\n\nQuestion: {question}')

try:
    response = ollama.chat(model=MODEL, messages=[{'role': 'user', 'content': prompt}])
    answer = response['message']['content']
except Exception as error:
    answer = (f'Ollama could not generate a response. Make sure Ollama is running and the model '
              f' is installed using: ollama pull {MODEL}.\nError: {error}')

print('\nFINAL RESPONSE FROM LLAMA 3.2\n', answer)

## Setup requirements

1. Install Ollama from https://ollama.com.
2. Start Ollama.
3. Run `ollama pull llama3.2` in a terminal.
4. Put your academic PDF beside this notebook and name it `academic_document.pdf`.
5. Run the notebook cells from top to bottom.

If the PDF is missing, the notebook automatically uses sample academic text.